In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

# Saving the Infidels

In this notebook we want so solve a famous search problem, which is usually known as the
[missionaries and cannibals problem](https://en.wikipedia.org/wiki/Missionaries_and_cannibals_problem):
Three missionaries and three infidels have to cross a river in order to get to a church where the infidels can be baptized.  In order to cross the river, they have to take a small boat that can take at most two passengers.  If at any moments at any shore there are more infidels than missionaries, then the missionaries have a problem, since the infidels have a diet that includes human flesh.

We will encode this problem as a *constraint satisfaction problem*.  In order to do so, we assume that the
problem can be solved with $n\in\mathbb{N}$ crossing of the river.  We use the following variables:
* $\texttt{M}i$ for $i\in\{0,\cdots,n\}$ is the number of missionaries on the western shore after the 
  $i^{\textrm{th}}$ crossing.
* $\texttt{C}i$ for $i\in\{0,\cdots,n\}$ is the number of infidels on the western shore after the 
  $i^{\textrm{th}}$ crossing.
* $\texttt{B}i$ for $i\in\{0,\cdots,n\}$ is the number of boats on the western shore after the 
  $i^{\textrm{th}}$ crossing.

In [ ]:
import_file "../z3_helpers.ml"

## Auxiliary Functions

The symbolic transition system uses 3 variables.
* `M` is the number of missionaries on the left side of the river,
* `C` is the number of infidels on the left side of the river,
* `B` is the number of boats on the left side of the river.

We define these variables using the function `int_const`.  The variables `MX`, `CX`, and `BX` describe the state after a crossing.  As the names of OCaml variables have to start with a lower case letter, the OCaml variables are called `m`, `c`, `b`, `mx`, `cx`, and `bx`.

In [ ]:
let m = int_const "M"  let mx = int_const "MX"
let c = int_const "C"  let cx = int_const "CX"
let b = int_const "B"  let bx = int_const "BX"

The function `start` takes three `Z3` variables as input:
* `m` is the number of missionaries on the western shore,
* `c` is the number of infidels on the western shore,
* `b` is the number of boats on the western shore.

It returns a `Z3` formula that specifies that everybody is on the western shore.
$$ \texttt{M} = 3 \wedge \texttt{C} = 3 \wedge \texttt{B} = 1 $$

In [ ]:
let start (m : Z3.Expr.expr) (c : Z3.Expr.expr) (b : Z3.Expr.expr) : Z3.Expr.expr =
  and_ [m =: int_val 3; c =: int_val 3; b =: int_val 1]

In [ ]:
start m c b

The function `goal` takes three `Z3` variables as input:
* `m` is the number of missionaries on the western shore,
* `c` is the number of infidels on the western shore,
* `b` is the number of boats on the western shore.

It returns a formula that specifies that everybody is on the eastern shore.
$$ \texttt{M} = 0 \wedge \texttt{C} = 0 \wedge \texttt{B} = 0 $$

In [ ]:
let goal (m : Z3.Expr.expr) (c : Z3.Expr.expr) (b : Z3.Expr.expr) : Z3.Expr.expr =
  and_ [m =: int_val 0; c =: int_val 0; b =: int_val 0]

The function `invariant` takes three `Z3` variables as input:
* `m` is the number of missionaries on the western shore,
* `c` is the number of infidels on the western shore,
* `b` is the number of boats on the western shore.

It returns a list of formulas.  The conjunction of these formulas is `true` if there is no problem on either shore of the river.
There is no problem if either of the following conditions is true:
* There are no missionaries on the western side of the shore, i.e.
  $\texttt{M} = 0$.
  Then all missionaries are on the eastern side of the shore.
* All missionaries are on the western side of the shore, i.e. $\texttt{M} = 3$.
  Then there are no missionaries on the eastern side of the shore.
* The number of missionaries on the western side is the same as the number of
  infidels on that side, i.e. $\texttt{M} = \texttt{C}$.  Then the numbers of
  missionaries and infidels have to match on the eastern shore as well.
* Furthermore, the number of missionaries and infidels is an element of the set $\{0,1,2,3\}$:
  $$ 0 \leq \texttt{M} \leq 3 \;\wedge\; 0 \leq \texttt{C} \leq 3$$

In [ ]:
let invariant (m : Z3.Expr.expr) (c : Z3.Expr.expr) (b : Z3.Expr.expr) : Z3.Expr.expr list =
  [ or_ [m =: int_val 0; m =: int_val 3; m =: c];   (* M = 0 ∨ M = 3 ∨ M = C *)
    m >=: int_val 0; m <=: int_val 3;               (* 0 ≤ M ≤ 3 *)
    c >=: int_val 0; c <=: int_val 3;               (* 0 ≤ C ≤ 3 *)
    b >=: int_val 0; b <=: int_val 1 ]              (* 0 ≤ B ≤ 1 *)

In [ ]:
List.iter (fun constr -> print_endline (Z3.Expr.to_string constr)) (invariant m c b)

The function `boat_ok` takes 6 arguments:
* `ma` is the number of missionaries on the western shore before the crossing.
* `ca` is the number of infidels on the western shore before the crossing.
* `ba` is the number of boats on the western shore before the crossing.
* `mb` is the number of missionaries on the western shore after the crossing.
* `cb` is the number of infidels on the western shore after the crossing.
* `bb` is the number of boats on the western shore after the crossing.

The function returns a formula that checks that if the boat
crosses the river from the western shore to the eastern shore, then the number of people on the boat is between $1$ and $2$
and the number of missionaries and infidels on the western shore can't increase:
$$ \mathtt{Ba} = 1 \;\rightarrow\; 1 \leq \mathtt{Ma} - \mathtt{Mb} + \mathtt{Ca} - \mathtt{Cb} \leq 2 \,\wedge\, \mathtt{Mb} \leq \mathtt{Ma} \,\wedge\, \mathtt{Cb} \leq \mathtt{Ca}$$
Here `Ma - Mb` is the number of missionaries on the boat and  `Ca - Cb` is the number of infidels on the boat.  The formula
$1 \leq \mathtt{Ma} - \mathtt{Mb} + \mathtt{Ca} - \mathtt{Cb} \leq 2$ expresses that there is at least one but not more than two passengers on the boat.  The parameter `bb` is not needed, but it makes the calls of `boat_ok` more uniform.

In [ ]:
let boat_ok (ma : Z3.Expr.expr) (ca : Z3.Expr.expr) (ba : Z3.Expr.expr)
            (mb : Z3.Expr.expr) (cb : Z3.Expr.expr) (bb : Z3.Expr.expr) : Z3.Expr.expr =
  ignore bb;
  let num_mis = ma -: mb in            (* Ma - Mb *)
  let num_can = ca -: cb in            (* Ca - Cb *)
  let all     = num_mis +: num_can in  (* Ma - Mb + Ca - Cb *)
  (* Ba = 1 → 1 ≤ Ma - Mb + Ca - Cb ≤ 2 ∧ Mb ≤ Ma ∧ Cb ≤ Ca *)
  implies (ba =: int_val 1)
          (and_ [all >=: int_val 1; all <=: int_val 2; mb <=: ma; cb <=: ca])

In [ ]:
boat_ok m c b mx cx bx

The function `transition` takes 6 arguments:
* `ma` is the number of missionaries on the western shore before the crossing.
* `ca` is the number of infidels on the western shore before the crossing.
* `ba` is the number of boats on the western shore before the crossing.
* `mb` is the number of missionaries on the western shore after the crossing.
* `cb` is the number of infidels on the western shore after the crossing.
* `bb` is the number of boats on the western shore after the crossing.

It returns a list of formulas that expresses the following:
* The boat changes the sides in every transition:
  $$ \mathtt{Bb} = 1 - \mathtt{Ba} $$
* If the boat crosses the river from the western shore to the eastern shore,
  then there is at least one and at most two passengers and the numbers on the western shore do not increase.
* If the boat crosses the river from the eastern shore to the western shore,
  then there is at least one and at most two passengers and the numbers on the eastern shore do not increase.

In [ ]:
let transition (ma : Z3.Expr.expr) (ca : Z3.Expr.expr) (ba : Z3.Expr.expr)
               (mb : Z3.Expr.expr) (cb : Z3.Expr.expr) (bb : Z3.Expr.expr) : Z3.Expr.expr list =
  [ bb =: int_val 1 -: ba;
    boat_ok ma ca ba mb cb bb;
    boat_ok mb cb bb ma ca ba ]

In [ ]:
transition m c b mx cx bx

The function `missionaries_csp` takes one argument:
- `n` is the number of crossings.

It creates a solver, adds all constraints (start, goal, invariants, transitions) for `n` crossings, and checks them.  If the constraints are satisfiable, it returns `Some solution`, where `solution` is a map that maps the names of the variables to their values.  Otherwise, it returns `None`.

In [ ]:
let missionaries_csp (n : int) : (string, int) RMap.t option =
  let solver = new_solver () in
  let ms = Array.init (n + 1) (fun i -> int_const (Printf.sprintf "M%d" i)) in
  let cs = Array.init (n + 1) (fun i -> int_const (Printf.sprintf "C%d" i)) in
  let bs = Array.init (n + 1) (fun i -> int_const (Printf.sprintf "B%d" i)) in
  Z3.Solver.add solver [start ms.(0) cs.(0) bs.(0)];
  Z3.Solver.add solver [goal ms.(n) cs.(n) bs.(n)];
  for i = 0 to n - 1 do
    Z3.Solver.add solver (invariant ms.(i) cs.(i) bs.(i));
    Z3.Solver.add solver (transition ms.(i) cs.(i) bs.(i) ms.(i + 1) cs.(i + 1) bs.(i + 1))
  done;
  match Z3.Solver.check solver [] with
  | Z3.Solver.SATISFIABLE ->
      let model = solver_model solver in
      Some (RMap.of_list
              (List.concat
                 (List.init (n + 1)
                    (fun i -> [(Printf.sprintf "M%d" i, eval_int model ms.(i));
                               (Printf.sprintf "C%d" i, eval_int model cs.(i));
                               (Printf.sprintf "B%d" i, eval_int model bs.(i))]))))
  | _ -> None

The function `find_solution` takes the argument `()`.  It computes a solution to the problem of saving the infidels by trying an increasing odd number of crossings.

In [ ]:
let find_solution () : (string, int) RMap.t =
  let rec try_crossings n =
    Printf.printf "Trying n=%d...\n" n;
    match missionaries_csp n with
    | Some solution -> solution
    | None          -> try_crossings (n + 2)
  in
  try_crossings 1

Solving the problem takes a fraction of a second.

In [ ]:
let solution = time "Missionaries Solving" find_solution

The function `repeat` takes two arguments:
- `s` is a string,
- `k` is a natural number.

It returns the string that consists of `k` copies of `s`.

The function `show_solution` takes one argument:
- `sol` is a solution, i.e. a map from the names of the variables to their values.

It prints the states and the passengers of the boat for every crossing.  As there are three variables for every state, the number of crossings is the number of variables divided by 3, minus 1.

In [ ]:
let repeat (s : string) (k : int) : string =
  String.concat "" (List.init k (fun _ -> s))

let show_solution (sol : (string, int) RMap.t) : unit =
  let n = RMap.cardinal sol / 3 - 1 in
  let get x i = RMap.find (Printf.sprintf "%s%d" x i) sol in
  for i = 0 to n do
    let m = get "M" i and c = get "C" i and b = get "B" i in
    let west = repeat "😇" m ^ repeat "🥷" c in
    let east = repeat "😇" (3 - m) ^ repeat "🥷" (3 - c) in
    print_endline (west ^ String.make 28 ' ' ^ east);
    if b = 1 && i < n then
      print_endline (String.make 12 ' ' ^ ">>> " ^ repeat "😇" (m - get "M" (i + 1)) ^ repeat "🥷" (c - get "C" (i + 1)) ^ " >>>")
    else if i < n then
      print_endline (String.make 12 ' ' ^ "<<< " ^ repeat "😇" (get "M" (i + 1) - m) ^ repeat "🥷" (get "C" (i + 1) - c) ^ " <<<")
  done

In [ ]:
show_solution solution

The function `show_animated_solution` takes one argument:
- `sol` is a solution, i.e. a map from the names of the variables to their values.

It displays an animation of the solution.  As in the notebook `05-Missionaries-STS.ipynb`, the HTML code is stored in the quoted string `template`, which contains the placeholders `__UID__`, `__N__`, and `__STATES__`.  The function `replace_all s pattern by` replaces every occurrence of the string `pattern` in `s` by the string `by`.

In [ ]:
#require "jupyter.notebook";;

In [ ]:
let replace_all (s : string) (pattern : string) (by : string) : string =
  let n = String.length s and k = String.length pattern in
  let buffer = Buffer.create n in
  let rec loop i =
    if i > n - k then
      Buffer.add_string buffer (String.sub s i (n - i))
    else if String.sub s i k = pattern then begin
      Buffer.add_string buffer by;
      loop (i + k)
    end else begin
      Buffer.add_char buffer s.[i];
      loop (i + 1)
    end
  in
  loop 0;
  Buffer.contents buffer

let template = {|
    <div style="font-family: sans-serif; max-width: 600px; margin: 0 auto; text-align: center;">
        <div style="margin-bottom: 15px;">
            <button id="btn-reset-__UID__" style="padding: 8px 16px; cursor: pointer;">Reset</button>
            <button id="btn-prev-__UID__" style="padding: 8px 16px; cursor: pointer;">Previous Step</button>
            <button id="btn-next-__UID__" style="padding: 8px 16px; cursor: pointer; font-weight: bold;">Next Step</button>
            <span id="label-step-__UID__" style="margin-left: 15px; font-weight: bold; font-size: 16px;">Step 0 of __N__</span>
        </div>
        
        <svg width="600" height="250" style="background-color: #e0f7fa; border: 1px solid #b2ebf2; border-radius: 8px;">
            <rect x="0" y="0" width="150" height="250" fill="#a5d6a7" />
            <rect x="450" y="0" width="150" height="250" fill="#a5d6a7" />
            
            <text x="75" y="30" font-size="14" font-weight="bold" fill="#2e7d32" text-anchor="middle">Western Shore</text>
            <text x="525" y="30" font-size="14" font-weight="bold" fill="#2e7d32" text-anchor="middle">Eastern Shore</text>

            <text id="west-m-__UID__" x="75" y="100" font-size="28" text-anchor="middle">😇😇😇</text>
            <text id="west-c-__UID__" x="75" y="150" font-size="28" text-anchor="middle">🥷🥷🥷</text>
            
            <text id="east-m-__UID__" x="525" y="100" font-size="28" text-anchor="middle"></text>
            <text id="east-c-__UID__" x="525" y="150" font-size="28" text-anchor="middle"></text>

            <g id="boat-__UID__" style="transition: transform 1.5s ease-in-out;">
                <rect x="160" y="160" width="80" height="40" rx="8" fill="#8d6e63" stroke="#5d4037" stroke-width="2"/>
                <text id="boat-p-__UID__" x="200" y="188" font-size="22" text-anchor="middle"></text>
            </g>
        </svg>
    </div>

    <script>
    (function() {
        const states = __STATES__;
        let currentStep = 0;
        let isAnimating = false;
        
        const boat = document.getElementById('boat-__UID__');
        const westM = document.getElementById('west-m-__UID__');
        const westC = document.getElementById('west-c-__UID__');
        const eastM = document.getElementById('east-m-__UID__');
        const eastC = document.getElementById('east-c-__UID__');
        const boatP = document.getElementById('boat-p-__UID__');
        const stepLabel = document.getElementById('label-step-__UID__');
        
        // Web Audio API context for the motor sound
        let audioCtx;
        function playMotorSound() {
            if (!audioCtx) audioCtx = new (window.AudioContext || window.webkitAudioContext)();
            if (audioCtx.state === 'suspended') audioCtx.resume();
            
            const osc = audioCtx.createOscillator();
            const filter = audioCtx.createBiquadFilter();
            const gain = audioCtx.createGain();
            
            // DIESEL ENGINE SIMULATION:
            // A sawtooth wave at ~12 Hz creates a rapid "clicking/knocking" 
            // sound that mimics the slow chug of a diesel engine.
            osc.type = 'sawtooth';
            osc.frequency.value = 12; 
            
            // Low-pass filter to muffle the harsh highs and keep it sounding bassy/heavy
            filter.type = 'lowpass';
            filter.frequency.value = 150;

            gain.gain.setValueAtTime(0, audioCtx.currentTime);
            // Using a slightly higher gain (0.6) because extreme low frequencies are perceived as quieter
            gain.gain.linearRampToValueAtTime(0.6, audioCtx.currentTime + 0.2); 
            
            osc.connect(filter);
            filter.connect(gain);
            gain.connect(audioCtx.destination);
            osc.start();
            
            return { osc, gain };
        }
        
        function stopMotorSound(sound) {
            if (sound) {
                sound.gain.gain.linearRampToValueAtTime(0, audioCtx.currentTime + 0.4); // Slower fade out for realism
                setTimeout(() => sound.osc.stop(), 400);
            }
        }

        function renderStaticState(step) {
            const s = states[step];
            westM.textContent = '😇'.repeat(s.M);
            westC.textContent = '🥷'.repeat(s.C);
            eastM.textContent = '😇'.repeat(3 - s.M);
            eastC.textContent = '🥷'.repeat(3 - s.C);
            boatP.textContent = '';
            
            boat.style.transform = s.B === 1 ? 'translateX(0px)' : 'translateX(200px)';
            stepLabel.textContent = 'Step ' + step + ' of __N__';
        }

        async function animateTransition(fromStep, toStep) {
            if (isAnimating) return;
            isAnimating = true;
            
            const s0 = states[fromStep];
            const s1 = states[toStep];
            
            const mPass = Math.abs(s0.M - s1.M);
            const cPass = Math.abs(s0.C - s1.C);
            
            if (s0.B === 1) { 
                westM.textContent = '😇'.repeat(s0.M - mPass);
                westC.textContent = '🥷'.repeat(s0.C - cPass);
            } else { 
                eastM.textContent = '😇'.repeat((3 - s0.M) - mPass);
                eastC.textContent = '🥷'.repeat((3 - s0.C) - cPass);
            }
            boatP.textContent = '😇'.repeat(mPass) + '🥷'.repeat(cPass);
            
            const sound = playMotorSound();
            boat.style.transform = s1.B === 1 ? 'translateX(0px)' : 'translateX(200px)';
            
            await new Promise(r => setTimeout(r, 1500));
            
            stopMotorSound(sound);
            currentStep = toStep;
            renderStaticState(currentStep);
            isAnimating = false;
        }

        document.getElementById('btn-next-__UID__').addEventListener('click', () => {
            if (currentStep < states.length - 1) animateTransition(currentStep, currentStep + 1);
        });

        document.getElementById('btn-prev-__UID__').addEventListener('click', () => {
            if (currentStep > 0) animateTransition(currentStep, currentStep - 1);
        });

        document.getElementById('btn-reset-__UID__').addEventListener('click', () => {
            if (!isAnimating) {
                currentStep = 0;
                renderStaticState(0);
            }
        });

        renderStaticState(0);
    })();
    </script>
    |}

let show_animated_solution (sol : (string, int) RMap.t) : unit =
  let n = RMap.cardinal sol / 3 - 1 in
  let get x i = RMap.find (Printf.sprintf "%s%d" x i) sol in
  let states =
    "[" ^ String.concat ", "
            (List.init (n + 1)
               (fun i -> Printf.sprintf "{\"M\": %d, \"C\": %d, \"B\": %d}" (get "M" i) (get "C" i) (get "B" i)))
    ^ "]"
  in
  let uid  = Printf.sprintf "%07x" (Random.bits ()) in
  let html = replace_all (replace_all (replace_all template "__UID__" uid) "__N__" (string_of_int n))
                         "__STATES__" states in
  ignore (Jupyter_notebook.display "text/html" html)

In [ ]:
show_animated_solution solution